In [1]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name
        self.uniprot_pattern = re.compile(
            r"(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})(?:-\d+)?"
        )
        self.ensembl_pattern = re.compile(
            r"ENS[A-Z]{0,10}(?:E|FM|G|GT|P|R|T)\d{6,}(?:\.\d+)?"
        )

    def _get_uniprot(self, accession):
        # use HW2_1
        endpoint = "https://rest.uniprot.org/uniprotkb/accessions"
        http_function = requests.get
        http_args = {"params": {"accessions": accession}, "timeout": 30}
        return http_function(endpoint, **http_args)

    def _get_ensembl(self, id):
        # use HW2_1
        endpoint = f"https://rest.ensembl.org/lookup/id/{id.split('.')[0]}"
        http_function = requests.get
        http_args = {"headers": {"Content-Type": "application/json"}, "timeout": 30}
        return http_function(endpoint, **http_args)

    def _uniprot_parse_response(self, resp: dict):
        # use HW2_1
        resp = resp.json()
        output = {}
        for val in resp["results"]:
            accession = val["primaryAccession"]
            output[accession] = {
                "organism": val["organism"]["scientificName"],
                "geneInfo": val["genes"],
                "sequenceInfo": val["sequence"],
                "type": "protein"
            }
        return output

    def _ensembl_parse_response(self, resp: dict):
        # use HW2_1
        resp = resp.json()
        id = resp["id"]
        output = {
            id: {
                "object_type": resp.get("object_type"),
                "assembly_name": resp.get("assembly_name"),
                "species": resp.get("species"),
                "db_type": resp.get("db_type"),
                "biotype": resp.get("biotype"),
                "display_name": resp.get("display_name"),
                "id": resp.get("id"),
                "description": resp.get("description"),
                "canonical_transcript": resp.get("canonical_transcript"),
                "source": resp.get("source")
            }
        }
        return output

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        # this function calls either _get_uniprot() or _get_ensembl()
        # an then _uniprot_parse_response() or _ensembl_parse_response()
        # outputs a dictionary with results (see example in test_data)
        output = {
            f"file_info_{id}": {
                "description": seq_description,
                "sequence": seq_sequence
            }
        }
        response = None
        try:
            if database == "uniprot":
                response = self._get_uniprot(id)
                response.raise_for_status()
                parsed = self._uniprot_parse_response(response)
            else:
                response = self._get_ensembl(id)
                response.raise_for_status()
                parsed = self._ensembl_parse_response(response)
            output[f"database_info_{id}"] = next(iter(parsed.values()))
        except requests.RequestException as error:
            if response is not None:
                try:
                    error_output = response.json()
                except ValueError:
                    error_output = str(error)
            else:
                error_output = str(error)
            output[f"database_info_{id}"] = error_output
        return output

    def seqkit_stats(self) -> dict:
        # this function calls seqkit via subprocess
        # if error arises, returns stderr and finishes the parser execution
        # if stats are collected, return the result (see example in test_data)

        try:
            seqkit = subprocess.run(
                ("seqkit", "stats", self.filename, "-a"),
                capture_output=True,
                text=True
            )
        except FileNotFoundError as error:
            return {"error": str(error)}
        if seqkit.stderr:
            return {"error": seqkit.stderr.strip()}
        seqkit_out = seqkit.stdout.strip().split("\n")
        prop_names = seqkit_out[0].split()[1:]
        prop_vals = seqkit_out[1].split()[1:]
        seq_info = dict(zip(prop_names, prop_vals))
        seqkit_result = {
            "fasta_seqkit_stat_info": seq_info,
            "fasta_type": seq_info["type"],
            "fasta_num_seqs": int(seq_info["num_seqs"].replace(",", ""))
        }
        return seqkit_result

    def biopython_parser(self, seqkit_result) -> dict:
        # accepts seqkit_result dict
        # depending on fasta file type selects regular expression
        # parses fasta file via biopython
        # iterates over sequences and calls relevant database via _access_databases()
        # returns final result (info about each sequence from FASTA file + info from database)
        if "fasta_type" not in seqkit_result:
            return seqkit_result
        if seqkit_result["fasta_type"] == "Protein":
            pattern = self.uniprot_pattern
            database = "uniprot"
        else:
            pattern = self.ensembl_pattern
            database = "ensembl"
        output = {"DB_name": database}
        warning = False
        for seq in SeqIO.parse(self.filename, "fasta"):
            match = pattern.search(seq.description)
            if match is None:
                warning = True
                continue
            output.update(
                self._access_database(
                    match.group(0),
                    database,
                    seq.description,
                    str(seq.seq)
                )
            )
        if warning:
            output["WARNING"] = {"No ID match found."}
        return output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [3]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '2',
  'sum_len': '456',
  'min_len': '29',
  'avg_len': '228',
  'max_len': '427',
  'Q1': '29',
  'Q2': '228',
  'Q3': '427',
  'sum_gap': '0',
  'N50': '427',
  'N50_num': '1',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0',
  'GC(%)': '0',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [4]:
biopython = parser.biopython_parser(stats)

parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS